# L3 课程笔记：Logistic 回归（logistic regression）与支持向量机（support vector machine, SVM）

> **资料对应关系。** 核心内容顺序遵循 L3 slides（第 2-100 页）。以下补充参考了 *Learning Theory from First Principles* 的第 1.1.5、4.1、12.1.2、13.1.1、14.2.1 节，以及 *Understanding Machine Learning: Solution Manual* 第 15 节中的间隔论证。



## 1. 过拟合（overfitting）、经验风险最小化（empirical risk minimization, ERM）与归纳偏置（inductive bias）

在经验风险最小化中，我们通过最小化训练损失来选择预测器：

$$
\operatorname{ERM}_{\mathcal H}(S)
\in
\operatorname*{argmin}_{h \in \mathcal H} L_S(h),
$$

其中 $S$ 是训练集，$\mathcal H$ 是选定的假设类。

关键问题是过拟合。如果 $\mathcal H$ 过大，模型可能很好地拟合训练数据，却在未见数据上泛化（generalization）不佳。因此，本讲的核心思想是

$$
\text{如果限制搜索空间 } \mathcal H,\text{ ERM 就可能有效}.
$$

这种限制称为**归纳偏置**（inductive bias）。本讲采用的归纳偏置是**线性模型**（linear model）。

---



## 2. 线性假设类（linear hypothesis class）

线性分类器（linear classifier）以超平面（hyperplane）为基础：

$$
w^\top x+w_0=0,
$$

其中：

- $w\in\mathbb R^d$ 是权重向量（weight vector）；
- $w_0\in\mathbb R$ 是偏置或截距（bias/intercept）；
- $x\in\mathbb R^d$ 是输入。

线性假设类可以写成

$$
\mathcal H=
\left\{
x\mapsto w^\top x+w_0
\;:\;
w\in\mathbb R^d,\; w_0\in\mathbb R
\right\}.
$$

对于标签 $y_i\in\{-1,+1\}$ 的二分类，分类器为

$$
h(x;w,w_0)=\operatorname{sign}(w^\top x+w_0).
$$

样本点 $(x_i,y_i)$ 被正确分类，当且仅当

$$
y_i(w^\top x_i+w_0)>0.
$$

量

$$
z_i=y_i(w^\top x_i+w_0)
$$

称为**带符号间隔得分**（signed margin score）。$z_i>0$ 表示分类正确，$z_i<0$ 表示分类错误。

---



## 3. 分类损失函数（classification loss function）

理想的分类损失是 0/1 损失（0/1 loss）：

$$
\ell_{0/1}(z)=\mathbf 1_{\{z\le 0\}}.
$$

但该损失非凸（nonconvex）且难以优化，因此通常使用凸上界（convex upper bound）代替它。

两个重要例子是 **Logistic 损失（logistic loss）**

$$
\ell_{\log}(z)=\log(1+e^{-z}),
$$

以及 **合页损失（hinge loss）**

$$
\ell_h(z)=\max\{0,1-z\}.
$$

它们满足

$$
\ell_{0/1}(z)\le \frac{1}{\log 2}\ell_{\log}(z)
\quad\text{且}\quad
\ell_{0/1}(z)\le \ell_h(z),
$$

这里采用的约定是 $\ell_{0/1}(z)=\mathbf 1_{\{z\le0\}}$，因此第一个不等式需要因子 $1/\log 2$。当 $z\le0$ 时，未经缩放的 logistic 损失至少为 $\log2$，而不是 $1$。这两个替代损失（surrogate losses）都惩罚分类错误，同时比直接优化 0/1 损失容易得多。

### 为什么最小化替代损失（surrogate loss）在统计上是合理的？

替代损失仅仅是一个上界还不够。令

$$
\eta(x)=P(Y=+1\mid X=x)
$$

并令 $u=g(x)$ 为实值评分。对于间隔损失（margin loss）$\Phi$，其条件替代风险（conditional surrogate risk）为

$$
C_\eta^\Phi(u)=\eta\Phi(u)+(1-\eta)\Phi(-u).
$$

直接最小化 logistic 条件风险得到

$$
u_{\log}^*=\log\frac{\eta}{1-\eta},
$$

而对于 hinge 损失，当 $\eta\neq1/2$ 时，

$$
u_h^*=\operatorname{sign}(2\eta-1).
$$

两种最优评分都与 $2\eta-1$ 同号，因此在零处进行阈值化后都得到 Bayes 分类器（Bayes classifier）。该性质称为**分类校准**（classification calibration）或 **Fisher 一致性**（Fisher consistency）。更一般地，一个在零点可微的凸间隔损失是分类校准的，当且仅当 $\Phi'(0)<0$。

这种关系还可以定量描述：对于 hinge 损失，超额 0/1 风险不超过超额 hinge 风险；对于 logistic 等光滑校准损失（smooth calibrated loss），标准校准不等式用超额替代风险（excess surrogate risk）平方根的常数倍控制超额 0/1 风险（excess 0/1 risk）。

---



## 4. 证明：0/1 损失等于误分类概率（misclassification probability）

对于分类器 $h$，定义

$$
\ell_{0/1}(h(x),y)=
\begin{cases}
1, & h(x)\neq y,\\
0, & h(x)=y.
\end{cases}
$$

总体风险为

$$
L(h)=\mathbb E[\ell_{0/1}(h(X),Y)].
$$

因为 $\ell_{0/1}(h(X),Y)$ 是指示随机变量（indicator random variable），

$$
\ell_{0/1}(h(X),Y)=\mathbf 1_{\{h(X)\neq Y\}}.
$$

所以

$$
L(h)=\mathbb E\left[\mathbf 1_{\{h(X)\neq Y\}}\right]
=\mathbb P(h(X)\neq Y).
$$

因此，最小化期望 0/1 损失与最小化误分类概率完全相同。

---



# 第一部分：Logistic 回归（logistic regression）

## 5. Bayes 分类器（Bayes classifier）

对于标签 $Y\in\{0,1\}$ 的二分类，定义

$$
\eta(x)=\mathbb P(Y=1\mid X=x).
$$

Bayes 分类器为

$$
h^*(x)=
\begin{cases}
1, & \eta(x)>\frac12,\\
0, & \text{其他情况}.
\end{cases}
$$

直观上，如果给定 $x$ 后类别 $1$ 比类别 $0$ 更可能出现，就预测类别 $1$。

---



## 6. 证明：Bayes 分类器是最优的

我们证明，对于任意分类器 $h:\mathbb R^d\to\{0,1\}$，

$$
\mathbb P(h^*(X)\neq Y)
\le
\mathbb P(h(X)\neq Y).
$$

对 $X=x$ 条件化。若预测为 $1$，条件错误概率为

$$
\mathbb P(Y=0\mid X=x)=1-\eta(x).
$$

若预测为 $0$，条件错误概率为

$$
\mathbb P(Y=1\mid X=x)=\eta(x).
$$

所以 $x$ 处最小的条件误差是

$$
\min\{\eta(x),1-\eta(x)\}.
$$

若 $\eta(x)>1/2$，则 $1-\eta(x)<\eta(x)$，预测 $1$ 更好；若 $\eta(x)\le1/2$，则 $\eta(x)\le1-\eta(x)$，预测 $0$ 更好。

因此，$h^*$ 在每个固定的 $x$ 处都最小化条件误差。对 $X$ 取期望得到

$$
\mathbb P(h^*(X)\neq Y)
\le
\mathbb P(h(X)\neq Y).
$$

所以 Bayes 分类器是最优的。

---



## 7. 为什么不能直接使用 $\hat\eta(x)=w^\top x+w_0$？

Bayes 分类器把 $\eta(x)$ 当作概率：

$$
\eta(x)=\mathbb P(Y=1\mid X=x).
$$

因此必须满足

$$
0\le \eta(x)\le 1.
$$

但线性函数

$$
w^\top x+w_0
$$

可以取从 $-\infty$ 到 $+\infty$ 的任意实数，所以不能直接表示概率。

我们需要一个把实数映射到 $[0,1]$ 的函数，这就是 logistic 回归使用 sigmoid 函数（sigmoid function）的原因。

---



## 8. Sigmoid 函数（sigmoid function）

sigmoid（或 logistic）函数为

$$
\sigma(z)=\frac{1}{1+e^{-z}}=\frac{e^z}{1+e^z}.
$$

它满足

$$
\sigma:\mathbb R\to(0,1).
$$

因此 logistic 回归建模为

$$
\hat\eta(x)=\mathbb P(Y=1\mid X=x)\approx \sigma(w^\top x+w_0).
$$

通常把偏置 $w_0$ 吸收到特征向量中。定义

$$
\tilde x=
\begin{bmatrix}
x\\
1
\end{bmatrix},
\qquad
\tilde w=
\begin{bmatrix}
w\\
w_0
\end{bmatrix}.
$$

则

$$
w^\top x+w_0=\tilde w^\top \tilde x.
$$

所以经常简写为 $\sigma(w^\top x)$。

下面三个恒等式尤其有用：

$$
\sigma(-z)=1-\sigma(z),
\qquad
\sigma'(z)=\sigma(z)(1-\sigma(z)),
$$

以及当 $p=\sigma(z)$ 时，

$$
\log\frac{p}{1-p}=z.
$$

因此，logistic 回归对于**对数优势比**（log-odds）是线性的：

$$
\log\frac{P(Y=1\mid x)}{P(Y=0\mid x)}=w^\top x+w_0.
$$

另外，$\sigma(z)\ge1/2$ 当且仅当 $z\ge0$，所以概率决策规则与线性评分具有相同的超平面决策边界（decision boundary）。

---



## 9. 通过最大似然（maximum likelihood）推导 logistic 回归

假设标签 $y_i\in\{0,1\}$，数据为

$$
S=\{(x_i,y_i)\}_{i=1}^N.
$$

logistic 回归假设

$$
p(y_i=1\mid x_i)=\sigma(w^\top x_i),
$$

并且

$$
p(y_i=0\mid x_i)=1-\sigma(w^\top x_i).
$$

因此，对于单个观测，

$$
p(y_i\mid x_i)=
\sigma(w^\top x_i)^{y_i}
\left(1-\sigma(w^\top x_i)\right)^{1-y_i}.
$$

假设样本独立同分布（independent and identically distributed, i.i.d.），似然函数（likelihood function）为

$$
\mathcal L(w)=
\prod_{i=1}^N
\sigma(w^\top x_i)^{y_i}
\left(1-\sigma(w^\top x_i)\right)^{1-y_i}.
$$

最大化似然等价于最小化负对数似然（negative log-likelihood, NLL）：

$$
L(w)=-\log\mathcal L(w).
$$

所以

$$
L(w)=
-\sum_{i=1}^N
\log
\left[
\sigma(w^\top x_i)^{y_i}
\left(1-\sigma(w^\top x_i)\right)^{1-y_i}
\right].
$$

使用对数运算规则，

$$
L(w)=
-\sum_{i=1}^N
\left[
y_i\log\sigma(w^\top x_i)
+
(1-y_i)\log(1-\sigma(w^\top x_i))
\right].
$$

这就是**交叉熵损失**（cross-entropy loss）。

---



## 10. 标签为 $\{-1,+1\}$ 时的 Logistic 损失（logistic loss）

若标签 $y_i\in\{-1,+1\}$，logistic 回归可以写成

$$
p(y_i\mid x_i)=\sigma(y_iw^\top x_i).
$$

负对数似然为

$$
L(w)=\sum_{i=1}^N -\log\sigma(y_iw^\top x_i).
$$

由于

$$
\sigma(z)=\frac{1}{1+e^{-z}},
$$

所以

$$
-\log\sigma(z)
=-\log\left(\frac{1}{1+e^{-z}}\right)
=\log(1+e^{-z}).
$$

因此

$$
L(w)=
\sum_{i=1}^N
\log\left(1+\exp(-y_iw^\top x_i)\right).
$$

logistic 损失即

$$
\ell_{\log}(z)=\log(1+e^{-z}),
$$

其中

$$
z_i=y_iw^\top x_i.
$$

对应的 ERM 形式为

$$
\min_w
\frac1N
\sum_{i=1}^N
\log\left(1+\exp(-y_iw^\top x_i)\right).
$$

---



## 11. Logistic 回归的梯度（gradient）

令

$$
L(w)=\sum_{i=1}^N \log(1+\exp(-y_iw^\top x_i)).
$$

定义

$$
z_i=y_iw^\top x_i.
$$

则

$$
\ell_{\log}(z_i)=\log(1+e^{-z_i}).
$$

求导得到

$$
\frac{d}{dz}\log(1+e^{-z})
=
\frac{-e^{-z}}{1+e^{-z}}
=
-\frac{1}{1+e^z}.
$$

又因为

$$
\nabla_w z_i=y_i x_i,
$$

所以

$$
\nabla_w L(w)=
-\sum_{i=1}^N
\frac{y_i x_i}{1+\exp(y_iw^\top x_i)}.
$$

对于 $y_i\in\{0,1\}$ 的形式，等价地有

$$
\nabla_w L(w)=
\sum_{i=1}^N
\left(\sigma(w^\top x_i)-y_i\right)x_i.
$$

### Hessian 矩阵（Hessian matrix）与凸性（convexity）

对于 $\{-1,+1\}$ 形式，令

$$
d_i=\sigma(y_iw^\top x_i)\sigma(-y_iw^\top x_i)>0
$$

并令 $D=\operatorname{Diag}(d_1,\ldots,d_N)$。若 $X$ 的各行为 $x_i^\top$，则

$$
\nabla^2 L(w)=X^\top D X
=\sum_{i=1}^N d_i x_i x_i^\top.
$$

对于平均损失，应在该式前乘 $1/N$。对任意向量 $v$，

$$
v^\top\nabla^2L(w)v=\sum_{i=1}^N d_i(x_i^\top v)^2\ge0,
$$

因此 logistic 回归是凸问题。当设计矩阵（design matrix）满列秩（full column rank）时，它严格凸（strictly convex）。加入 L2 惩罚会在被惩罚坐标上增加一个单位阵的正倍数，因此即使 $X$ 秩亏（rank deficient），也可以获得强凸性（strong convexity）。

---



## 12. 数据线性可分（linearly separable）时会发生什么？

假设数据线性可分，则存在向量 $u$ 使得

$$
y_i u^\top x_i>0
\quad
\text{对所有 } i.
$$

考虑用正标量 $c$ 缩放 $u$：

$$
w=cu.
$$

于是

$$
y_iw^\top x_i=c y_i u^\top x_i.
$$

当 $c\to\infty$ 时，

$$
y_iw^\top x_i\to\infty.
$$

所以每个 logistic 损失项满足

$$
\log(1+\exp(-y_iw^\top x_i))\to\log(1+0)=0.
$$

因此

$$
L(cu)\to 0
\quad\text{当}\quad
c\to\infty.
$$

但对于有限的 $c$，每一项仍严格为正：

$$
\log(1+\exp(-y_iw^\top x_i))>0.
$$

所以在严格线性可分数据上，logistic 回归没有有限的最小化器。损失趋近于下确界 $0$，但只有在

$$
\|w\|\to\infty
$$

时才能实现这一极限。这就是 slides 中提到的“无穷大问题”。

### 梯度下降（gradient descent）的隐式最大间隔偏置（implicit max-margin bias）

不存在有限最小化器，并不意味着迭代没有结构。在标准的可分性与步长条件下，无正则化梯度下降满足

$$
\|w_t\|\to\infty,
\qquad
\frac{w_t}{\|w_t\|}\to\frac{w_{\mathrm{mm}}}{\|w_{\mathrm{mm}}\|},
$$

其中 $w_{\mathrm{mm}}$ 是 L2 最大间隔分离器，等价地是下列问题的一个解：

$$
\min_w\frac12\|w\|^2
\quad\text{满足}\quad
y_iw^\top x_i\ge1\ \text{ 对所有 }i.
$$

因此，参数范数发散，但方向收敛到硬间隔 SVM（hard-margin SVM）的方向。这是优化算法产生的**隐式正则化**（implicit regularization）。

---



## 13. 正则化 Logistic 回归（regularized logistic regression）

为了解决参数范数发散的问题，加入正则项（regularizer）：

$$
\min_w
\sum_{i=1}^N
\log\left(1+\exp(-y_iw^\top x_i)\right)
+
\lambda\|w\|_q^q.
$$

常见选择包括

$$
q=2
\quad\Rightarrow\quad
\lambda\|w\|_2^2,
$$

称为 **L2 正则化（L2 regularization）logistic 回归**；以及

$$
q=1
\quad\Rightarrow\quad
\lambda\|w\|_1,
$$

称为 **L1 正则化（L1 regularization）稀疏 logistic 回归**。

L2 形式尤其方便，因为

$$
\lambda\|w\|_2^2
$$

在 $\lambda>0$ 时使目标函数关于 $w$ 强凸。强凸性改善优化性质，并通常带来更稳定的解。

一种常见的归一化写法（normalized convention）是

$$
\min_{w,w_0}\frac1N\sum_{i=1}^N
\log\bigl(1+e^{-y_i(w^\top x_i+w_0)}\bigr)
+\frac\lambda2\|w\|_2^2.
$$

截距 $w_0$ 通常不被惩罚。因此，强凸性在被惩罚的 $w$ 坐标上得到保证；完整增广参数是否唯一，还取决于数据和截距约定。

正则化也有概率解释：使用 $w$ 的各向同性高斯先验（isotropic Gaussian prior）进行最大后验估计（maximum a posteriori estimation, MAP），会得到 L2 惩罚；使用拉普拉斯先验（Laplace prior）会得到 L1 惩罚。因此，正则化既可以理解为容量控制（capacity control），也可以理解为对参数的先验偏好（prior preference）。

---



# 第二部分：多分类 Logistic 回归（multiclass logistic regression）

对于 $K$ 个类别，$Y\in\{0,1,\dots,K-1\}$，二元 logistic 回归推广为 **Softmax 回归（softmax regression）**。

每个类别 $k$ 有一个权重向量 $w_k$，模型为

$$
p(Y=k\mid X=x)=
\frac{\exp(w_k^\top x)}
{\sum_{j=0}^{K-1}\exp(w_j^\top x)}.
$$

分类器预测概率最大的类别：

$$
h(x)=\operatorname*{argmax}_{k} p(Y=k\mid X=x).
$$

由于分母对所有类别相同，

$$
h(x)=\operatorname*{argmax}_{k} w_k^\top x.
$$

对于独热编码（one-hot encoding）标签 $y_{ik}$，交叉熵损失为

$$
L(W)=
-\sum_{i=1}^N
\sum_{k=0}^{K-1}
y_{ik}
\log p(Y=k\mid X=x_i).
$$

### 不变性（invariance）与可识别性（identifiability）

如果给每个类别权重都加上同一个向量 $a$，softmax 概率不变：

$$
w_k\mapsto w_k+a
\quad\Longrightarrow\quad
w_k^\top x\mapsto w_k^\top x+a^\top x.
$$

新增项对所有对数几率得分（logits）都相同，因此在分子和分母之间抵消。所以不受约束的参数化（parameterization）不可识别。两个标准约定是：把一个参考类别（reference class）的参数设为零，或者要求 $\sum_k w_k=0$；二者都不会改变拟合概率或预测结果。

### 梯度（gradient）与数值稳定性（numerical stability）

对于类别为 $y_i$ 的单个观测，softmax 损失为

$$
\ell_i(W)=-w_{y_i}^\top x_i+
\log\sum_{j=0}^{K-1}e^{w_j^\top x_i}.
$$

若 $p_{ik}=P(Y=k\mid X=x_i)$，则

$$
\frac{\partial\ell_i}{\partial w_k}
=\bigl(p_{ik}-\mathbf 1_{\{y_i=k\}}\bigr)x_i.
$$

为保证数值稳定，应在计算对数和指数（log-sum-exp）前减去 $m_i=\max_j w_j^\top x_i$：

$$
\log\sum_j e^{w_j^\top x_i}
=m_i+\log\sum_j e^{w_j^\top x_i-m_i}.
$$

### 与生成模型（generative model）的联系

假设 $P(Y=k)=\pi_k$，并且各类别的条件输入服从具有共同协方差矩阵（shared covariance matrix）的高斯分布（Gaussian distribution）：

$$
X\mid Y=k\sim\mathcal N(\mu_k,\Sigma).
$$

利用 Bayes 公式得到 softmax 模型，其中

$$
w_k=\Sigma^{-1}\mu_k,
\qquad
b_k=\log\pi_k-\frac12\mu_k^\top\Sigma^{-1}\mu_k.
$$

这就是线性判别分析（linear discriminant analysis, LDA）。它说明线性 softmax 评分既可以通过条件似然优化以判别式（discriminative）方式得到，也可以从共享协方差的 Gaussian 类别以生成式（generative）方式得到。当 $K=2$ 时，它退化为二元 logistic 回归。

---



# 第三部分：支持向量机（support vector machine, SVM）

## 14. 从合页损失（hinge loss）到 SVM

SVM 使用 hinge 损失

$$
\ell_h(z)=\max\{0,1-z\},
$$

其中

$$
z_i=y_i(w^\top x_i+w_0).
$$

SVM 目标函数并不只有 hinge 损失，还包含关键的正则项（regularizer）

$$
\frac12\|w\|^2.
$$

因此，软间隔 SVM（soft-margin SVM）的目标为

$$
\min_{w,w_0}
\frac12\|w\|^2
+
C\sum_{i=1}^N
\max\{0,1-y_i(w^\top x_i+w_0)\}.
$$

正则项控制间隔，hinge 损失控制分类错误和间隔违反。

---



## 15. 凸包（convex hull）的几何直观

对于两个类别，分别考虑正类样本的凸包和负类样本的凸包。

如果两个凸包不相交，就存在一个超平面将它们分开。从几何上看，SVM 在所有分离超平面中选择到两类最近点距离最大的一个。

最靠近分离超平面的点成为**支持向量**（support vectors）。

更精确地，令 $p_+$ 和 $p_-$ 是两个凸包中距离最近的一对点。最大间隔分离器（maximum-margin separator）与 $p_+-p_-$ 正交，并位于两个支撑超平面（supporting hyperplanes）的正中间。两个凸包之间的距离为 $\|p_+-p_-\|$，因此到任一侧的几何间隔（geometric margin）是该距离的一半。

该图像直接适用于线性可分的硬间隔（hard-margin）情形。若两个凸包相交，则不存在 hard-margin 分离器；软间隔（soft-margin）形式在间隔大小与违反量之间权衡。在这种情况下，支持向量可能位于间隔边界上、间隔内部，甚至决策边界的错误一侧。

---



## 16. 规范化尺度（canonical normalization）

假设数据严格线性可分，则存在 $(w,w_0)$ 使得

$$
y_i(w^\top x_i+w_0)>0
\quad
\text{对所有 } i.
$$

如果 $(w,w_0)$ 能分离数据，那么对任意 $\delta>0$，

$$
(\delta w,\delta w_0)
$$

也能分离数据，因为

$$
y_i(\delta w^\top x_i+\delta w_0)
=
\delta y_i(w^\top x_i+w_0)>0.
$$

所以 $(w,w_0)$ 的尺度是任意的。为消除这种歧义，SVM 使用**规范化尺度**（canonical normalization）：

$$
\min_{1\le i\le N}|w^\top x_i+w_0|=1.
$$

对于标签 $y_i\in\{-1,+1\}$，等价地，

$$
\min_{1\le i\le N}y_i(w^\top x_i+w_0)=1.
$$

因此所有训练点都满足

$$
y_i(w^\top x_i+w_0)\ge 1.
$$

---



## 17. 证明：点到超平面的距离（point-to-hyperplane distance）

令超平面为

$$
H=\{x:w^\top x+w_0=0\}.
$$

任意点 $x$ 到 $H$ 的距离为

$$
\operatorname{dist}(x,H)=\frac{|w^\top x+w_0|}{\|w\|}.
$$

证明如下。

令 $x_H$ 为 $x$ 在超平面上的投影（projection）。由于 $x-x_H$ 与超平面垂直，它必与 $w$ 平行，所以

$$
x_H=x-\alpha w
$$

其中 $\alpha$ 是某个标量。

因为 $x_H$ 位于超平面上，

$$
w^\top x_H+w_0=0.
$$

代入 $x_H=x-\alpha w$：

$$
w^\top(x-\alpha w)+w_0=0.
$$

即

$$
w^\top x-\alpha\|w\|^2+w_0=0.
$$

解得

$$
\alpha=\frac{w^\top x+w_0}{\|w\|^2}.
$$

因此

$$
\|x-x_H\|=\|\alpha w\|=|\alpha|\|w\|=\frac{|w^\top x+w_0|}{\|w\|}.
$$

距离公式得证。

在规范化尺度下，最接近超平面的训练点满足

$$
|w^\top x_i+w_0|=1.
$$

所以它们到分离超平面的距离为

$$
\frac{1}{\|w\|}.
$$

因此间隔为

$$
\gamma=\frac{1}{\|w\|}.
$$

两个支撑超平面

$$
w^\top x+w_0=1
$$

与

$$
w^\top x+w_0=-1
$$

之间的距离为

$$
\frac{2}{\|w\|}.
$$

---



## 18. 为什么偏好大间隔（large margin）？

假设测试点由训练点经过扰动产生：

$$
(x,y)\mapsto(x+\delta x,y),
$$

且扰动有界：

$$
\|\delta x\|\le r.
$$

如果分类器的间隔 $\gamma>r$，则该扰动无法越过决策边界，所以扰动后的点仍被正确分类。

证明如下。对于正确分类的训练点，

$$
y(w^\top x+w_0)\ge \gamma\|w\|.
$$

对扰动后的点，

$$
y(w^\top(x+\delta x)+w_0)
=
y(w^\top x+w_0)+y w^\top\delta x.
$$

根据柯西–施瓦茨不等式（Cauchy–Schwarz inequality），

$$
|w^\top\delta x|
\le
\|w\|\|\delta x\|
\le
r\|w\|.
$$

因此

$$
y(w^\top(x+\delta x)+w_0)
\ge
\gamma\|w\|-r\|w\|
=
(\gamma-r)\|w\|.
$$

若 $\gamma>r$，则

$$
(\gamma-r)\|w\|>0.
$$

所以扰动后的点仍被正确分类。这解释了大间隔分类器的鲁棒性（robustness）直观。

---



## 19. 硬间隔 SVM（hard-margin SVM）

对于线性可分数据，我们希望最大化间隔

$$
\gamma=\frac{1}{\|w\|}.
$$

最大化 $1/\|w\|$ 等价于最小化 $\|w\|$。为便于数学处理，SVM 最小化

$$
\frac12\|w\|^2.
$$

因此 hard-margin SVM 为

$$
\min_{w,w_0}
\frac12\|w\|^2
$$

满足约束

$$
y_i(w^\top x_i+w_0)\ge 1,
\qquad
i=1,\dots,N.
$$

这是凸优化问题（convex optimization problem），因为 $\frac12\|w\|^2$ 是凸函数，每个约束都是仿射约束（affine constraint）。

### 对偶问题（dual problem）与支持向量条件

为间隔约束引入拉格朗日乘子（Lagrange multiplier）$\alpha_i\ge0$。hard-margin 对偶问题为

$$
\max_{\alpha\ge0}
\sum_{i=1}^N\alpha_i
-\frac12\sum_{i=1}^N\sum_{j=1}^N
\alpha_i\alpha_jy_iy_jx_i^\top x_j
$$

满足

$$
\sum_{i=1}^N\alpha_i y_i=0.
$$

拉格朗日函数（Lagrangian）的驻点条件（stationarity condition）给出

$$
w=\sum_{i=1}^N\alpha_i y_i x_i.
$$

互补松弛条件（complementary slackness condition）为

$$
\alpha_i\bigl(y_i(w^\top x_i+w_0)-1\bigr)=0.
$$

所以只有当点的间隔约束活跃，即 $y_i(w^\top x_i+w_0)=1$ 时，才可能有 $\alpha_i>0$。这些活跃观测就是支持向量。对偶问题只通过内积（inner product）$x_i^\top x_j$ 依赖数据，这也是核方法（kernel method）的入口。

---



## 20. 软间隔 SVM（soft-margin SVM）

如果数据不是线性可分的，hard-margin 约束可能不可行。引入松弛变量（slack variable）$\xi_i\ge0$：

$$
y_i(w^\top x_i+w_0)\ge 1-\xi_i.
$$

soft-margin SVM 为

$$
\min_{w,w_0,\xi}
\frac12\|w\|^2
+
C\sum_{i=1}^N \xi_i
$$

满足

$$
y_i(w^\top x_i+w_0)\ge 1-\xi_i,
\qquad
i=1,\dots,N,
$$

以及

$$
\xi_i\ge0.
$$

解释如下：

- $\xi_i=0$：该点满足间隔约束；
- $0<\xi_i<1$：该点分类正确，但位于间隔内部；
- $\xi_i\ge1$：该点被误分类，或已经到达/越过错误一侧。

参数 $C>0$ 控制权衡：

- 较大的 $C$：更强烈地惩罚违反，更接近 hard-margin SVM；
- 较小的 $C$：允许更多违反，通常得到更大的间隔。

即使样本线性可分，有限 $C$ 的 soft-margin 解也未必等于 hard-margin 解。当 $C$ 足够小时，接受某个间隔违反来换取显著更小的 $\|w\|$ 可能更加划算。因此，“数据可分”与“选择哪种优化形式”是两个不同事实。

在 soft-margin 对偶中，hard-margin 的条件 $\alpha_i\ge0$ 变为盒约束（box constraint）

$$
0\le\alpha_i\le C.
$$

KKT 条件（Karush–Kuhn–Tucker conditions）给出三种有用情形：

- $\alpha_i=0$：$y_i(w^\top x_i+w_0)\ge1$，该点在间隔外或边界上，并且不活跃；
- $0<\alpha_i<C$：$y_i(w^\top x_i+w_0)=1$，该点恰好位于间隔边界上；
- $\alpha_i=C$：$y_i(w^\top x_i+w_0)\le1$，该点位于间隔内部或被误分类。

---



## 21. 证明：soft-margin SVM 等价于合页损失形式（hinge-loss formulation）

从松弛变量形式出发：

$$
\min_{w,w_0,\xi}
\frac12\|w\|^2
+
C\sum_{i=1}^N \xi_i
$$

满足

$$
y_i(w^\top x_i+w_0)\ge 1-\xi_i,
$$

以及

$$
\xi_i\ge0.
$$

重排间隔约束：

$$
\xi_i\ge 1-y_i(w^\top x_i+w_0).
$$

结合 $\xi_i\ge0$，得到

$$
\xi_i\ge\max\{0,1-y_i(w^\top x_i+w_0)\}.
$$

固定 $w,w_0$ 后，目标函数关于每个 $\xi_i$ 单调递增，所以最优松弛变量取最小可行值：

$$
\xi_i^*=\max\{0,1-y_i(w^\top x_i+w_0)\}.
$$

将其代回目标函数：

$$
\min_{w,w_0}
\frac12\|w\|^2
+
C\sum_{i=1}^N
\max\{0,1-y_i(w^\top x_i+w_0)\}.
$$

这正是 hinge-loss SVM 的形式。

---



## 22. Logistic 回归与 SVM 的比较（comparison）

两种方法都使用线性评分

$$
w^\top x+w_0.
$$

但它们使用不同损失。

Logistic 回归：

$$
\min_w
\sum_{i=1}^N
\log(1+\exp(-y_iw^\top x_i))
+
\lambda\|w\|^2.
$$

SVM：

$$
\min_w
\frac12\|w\|^2
+
C\sum_{i=1}^N
\max\{0,1-y_iw^\top x_i\}.
$$

主要区别：

- logistic 回归通过 $\sigma(w^\top x)$ 给出概率输出；
- SVM 关注最大化间隔；
- logistic 损失光滑；
- hinge 损失不光滑；在 SVM 对偶中，只有 $\alpha_i>0$ 的观测出现在 $w$ 中；
- 对于线性模型和凸正则化，两者都是凸问题。

SVM 的稀疏性（sparsity）指对偶系数（dual coefficients）的稀疏，而不一定意味着支持向量数量很少。在高维问题中，支持向量个数仍可能与样本量成正比。

---



## 23. 需要记住的关键公式

线性分类器：

$$
h(x)=\operatorname{sign}(w^\top x+w_0).
$$

正确分类条件：

$$
y_i(w^\top x_i+w_0)>0.
$$

Logistic sigmoid：

$$
\sigma(z)=\frac{1}{1+e^{-z}}.
$$

交叉熵损失：

$$
L(w)=
-\sum_{i=1}^N
\left[
y_i\log\sigma(w^\top x_i)
+
(1-y_i)\log(1-\sigma(w^\top x_i))
\right].
$$

标签 $y_i\in\{-1,+1\}$ 时的 logistic 损失：

$$
L(w)=
\sum_{i=1}^N
\log(1+\exp(-y_iw^\top x_i)).
$$

Hinge 损失：

$$
\ell_h(z)=\max\{0,1-z\}.
$$

点到超平面的距离：

$$
\operatorname{dist}(x,H)=\frac{|w^\top x+w_0|}{\|w\|}.
$$

Hard-margin SVM：

$$
\min_{w,w_0}
\frac12\|w\|^2
\quad
\text{满足}
\quad
y_i(w^\top x_i+w_0)\ge1.
$$

Soft-margin SVM：

$$
\min_{w,w_0}
\frac12\|w\|^2
+
C\sum_{i=1}^N
\max\{0,1-y_i(w^\top x_i+w_0)\}.
$$

---



## 24. Slides 中主要问题的答案

**问题：为什么直接写 $\hat\eta(x)=w^\top x+w_0$ 不可行？**

因为 $\eta(x)$ 是概率，必须位于 $[0,1]$，但 $w^\top x+w_0$ 可以是任意实数。logistic 回归使用 sigmoid 修正这一问题：

$$
\hat\eta(x)=\sigma(w^\top x+w_0).
$$

**问题：数据线性可分时，logistic 回归会怎样？**

通过把 $w$ 缩放到无穷大，可以使损失任意接近 $0$：

$$
w=cu,
\qquad
c\to\infty.
$$

因此，无正则化问题没有有限最小化器。

**问题：为什么加入正则化？**

正则化控制模型复杂度，并防止 $\|w\|$ 趋向无穷：

$$
L_{\text{reg}}(w)=L(w)+\lambda\|w\|_q^q.
$$

**问题：为什么最靠近分离超平面的点距离为 $1/\|w\|$？**

在规范化尺度下，最近训练点满足

$$
|w^\top x_i+w_0|=1.
$$

又因为

$$
\operatorname{dist}(x_i,H)=\frac{|w^\top x_i+w_0|}{\|w\|},
$$

所以最近距离为

$$
\frac1{\|w\|}.
$$

**问题：hinge 损失如何从松弛变量中出现？**

最优松弛变量为

$$
\xi_i^*=\max\{0,1-y_i(w^\top x_i+w_0)\}.
$$

将其代入带松弛变量的 SVM，就得到 hinge-loss SVM。

